# Baseline Fine-Tuning — ViT-B/16 on CheXpert

Trains ViT-B/16 (ImageNet-21k) on CheXpert with weighted BCE.
Runs 3 seeds, saves best checkpoint per seed, reports mean AUC per pathology.

**On MacBook**: auto-uses `train_dev5k.csv`, 2 epochs, batch 8, fp32 — for pipeline validation only.
**On Colab**: auto-uses `train_full.csv`, 10 epochs, batch 32, bf16 — real training run.

In [1]:
# ── Colab-only: mount Drive and clone repo ────────────────────────────────────
import sys, os

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/dizertatie_project'):
        !git clone https://github.com/daviDpaD18/diz.git /content/diz
        os.symlink('/content/diz/dizertatie_project', '/content/dizertatie_project')
    sys.path.insert(0, '/content/dizertatie_project')
except ImportError:
    IS_COLAB = False
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

print('Colab:', IS_COLAB)

Colab: False


In [2]:
import json, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

from src.config import IMAGE_ROOT, SPLITS_DIR, WEIGHTS_DIR, CKPT_DIR
from src.dataset import LABEL_COLS, CheXpertDataset, train_transforms, eval_transforms

CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    AMP_DTYPE = torch.bfloat16   # bf16 on A100
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    AMP_DTYPE = None             # fp32 on MPS — fp16 unstable
else:
    DEVICE = torch.device('cpu')
    AMP_DTYPE = None

USE_AMP = AMP_DTYPE is not None
print(f'Device: {DEVICE}  |  AMP: {USE_AMP} ({AMP_DTYPE})')

Device: mps  |  AMP: False (None)


In [3]:
# ── Hyperparameters (auto-set per environment) ────────────────────────────────
TRAIN_CSV    = SPLITS_DIR / ('train_full.csv'  if IS_COLAB else 'train_dev5k.csv')
VALID_CSV    = SPLITS_DIR / 'valid_frontal.csv'
NUM_EPOCHS   = 10 if IS_COLAB else 2
BATCH_SIZE   = 32 if IS_COLAB else 8
NUM_WORKERS  = 4  if IS_COLAB else 0
LR           = 1e-4
WEIGHT_DECAY = 1e-2
WARMUP_RATIO = 0.05    # 5% of total training steps
GRAD_CLIP    = 1.0
SEEDS        = [42, 123, 456]
MODEL_NAME   = 'vit_base_patch16_224.augreg_in21k'

print(f'Train CSV : {TRAIN_CSV.name}  ({pd.read_csv(TRAIN_CSV).shape[0]:,} rows)')
print(f'Epochs    : {NUM_EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LR}')

Train CSV : train_dev5k.csv  (5,000 rows)
Epochs    : 2  |  Batch: 8  |  LR: 0.0001


In [4]:
# ── Class weights ─────────────────────────────────────────────────────────────
with open(WEIGHTS_DIR / 'class_weights.json') as f:
    cw = json.load(f)

CLASS_WEIGHTS = torch.tensor(
    [cw[l] for l in LABEL_COLS], dtype=torch.float32
).to(DEVICE)

print('Class weights loaded:')
for l, w in zip(LABEL_COLS, CLASS_WEIGHTS.tolist()):
    print(f'  {l:<35} {w:.4f}')

Class weights loaded:
  No Finding                          0.6603
  Enlarged Cardiomediastinum          1.2205
  Cardiomegaly                        0.4793
  Lung Opacity                        0.1190
  Lung Lesion                         1.5925
  Edema                               0.1823
  Consolidation                       0.3011
  Pneumonia                           2.4064
  Atelectasis                         0.1881
  Pneumothorax                        0.6335
  Pleural Effusion                    0.1296
  Pleural Other                       4.4751
  Fracture                            1.5073
  Support Devices                     0.1050


In [5]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model() -> torch.nn.Module:
    """Load ViT-B/16 with ImageNet-21k weights, replace head with 14-output linear."""
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=14)
    return model.to(DEVICE)


def weighted_bce(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Per-label weighted binary cross-entropy, averaged over labels and batch."""
    losses = F.binary_cross_entropy_with_logits(logits, labels, reduction='none')  # [B, 14]
    return (losses * CLASS_WEIGHTS).mean()


@torch.no_grad()
def evaluate(model, loader) -> tuple[float, dict]:
    """Run validation, return (macro_auc, per_label_auc_dict)."""
    model.eval()
    all_probs, all_labels = [], []

    for imgs, labels, *_ in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        all_probs.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels)

    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    auc = {}
    for i, label in enumerate(LABEL_COLS):
        if labels[:, i].sum() > 0:
            auc[label] = roc_auc_score(labels[:, i], probs[:, i])
        else:
            auc[label] = float('nan')   # no positives in val set for this label

    macro = float(np.nanmean(list(auc.values())))
    return macro, auc

In [6]:
# ── Data loaders ──────────────────────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_CSV)
df_valid = pd.read_csv(VALID_CSV)

val_ds     = CheXpertDataset(df_valid, IMAGE_ROOT, eval_transforms)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS,
                        shuffle=False, pin_memory=IS_COLAB)

print(f'Train: {len(df_train):,}  |  Val: {len(df_valid):,}')

Train: 5,000  |  Val: 202


In [7]:
# ── Training loop ─────────────────────────────────────────────────────────────
all_seed_results = {}

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'SEED {seed}')
    print(f'{"="*60}')

    set_seed(seed)

    # Fresh model and data loader per seed
    model = build_model()

    train_ds = CheXpertDataset(df_train, IMAGE_ROOT, train_transforms)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=IS_COLAB, drop_last=True,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )

    total_steps  = NUM_EPOCHS * len(train_loader)
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler() if USE_AMP else None

    best_auc  = 0.0
    best_ckpt = CKPT_DIR / f'baseline_seed{seed}_best.pt'

    for epoch in range(1, NUM_EPOCHS + 1):
        # ── Train ──
        model.train()
        running_loss = 0.0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False)
        for imgs, labels, *_ in pbar:
            imgs   = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            if USE_AMP:
                with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE):
                    logits = model(imgs)
                    loss   = weighted_bce(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(imgs)
                loss   = weighted_bce(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            scheduler.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}')

        avg_loss = running_loss / len(train_loader)

        # ── Validate ──
        macro_auc, auc_per_label = evaluate(model, val_loader)

        print(f'Epoch {epoch:>2} | loss {avg_loss:.4f} | macro AUC {macro_auc:.4f}'
              + (' ← best' if macro_auc > best_auc else ''))

        if macro_auc > best_auc:
            best_auc = macro_auc
            torch.save({
                'seed':              seed,
                'epoch':             epoch,
                'model_state_dict':  model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_macro_auc':     macro_auc,
                'val_auc_per_label': auc_per_label,
                'hparams': {
                    'model':       MODEL_NAME,
                    'lr':          LR,
                    'weight_decay': WEIGHT_DECAY,
                    'batch_size':  BATCH_SIZE,
                    'epochs':      NUM_EPOCHS,
                    'warmup_ratio': WARMUP_RATIO,
                },
            }, best_ckpt)
            print(f'         Checkpoint saved → {best_ckpt.name}')

    all_seed_results[seed] = {'macro_auc': best_auc, 'auc_per_label': auc_per_label}
    print(f'Seed {seed} done. Best macro AUC: {best_auc:.4f}')


SEED 42


Epoch  1 | loss 0.2094 | macro AUC 0.6220 ← best
         Checkpoint saved → baseline_seed42_best.pt


Epoch  2 | loss 0.1784 | macro AUC 0.6224 ← best
         Checkpoint saved → baseline_seed42_best.pt
Seed 42 done. Best macro AUC: 0.6224

SEED 123


Epoch  1 | loss 0.1996 | macro AUC 0.6102 ← best
         Checkpoint saved → baseline_seed123_best.pt


Epoch  2 | loss 0.1764 | macro AUC 0.6547 ← best
         Checkpoint saved → baseline_seed123_best.pt
Seed 123 done. Best macro AUC: 0.6547

SEED 456


Epoch  1 | loss 0.1977 | macro AUC 0.6491 ← best
         Checkpoint saved → baseline_seed456_best.pt


Epoch  2 | loss 0.1769 | macro AUC 0.6319
Seed 456 done. Best macro AUC: 0.6491


In [8]:
# ── Results summary across seeds ──────────────────────────────────────────────
import warnings
print('\n' + '='*60)
print('BASELINE RESULTS — mean ± std across 3 seeds')
print('='*60)

rows = []
for label in LABEL_COLS:
    vals = [all_seed_results[s]['auc_per_label'].get(label, float('nan')) for s in SEEDS]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        mean_auc = np.nanmean(vals)
        std_auc  = np.nanstd(vals)
    rows.append({'Label': label, 'Mean AUC': mean_auc, 'Std': std_auc})

results_df = pd.DataFrame(rows).set_index('Label')
macro_means = [all_seed_results[s]['macro_auc'] for s in SEEDS]

print(results_df.round(4).to_string())
print(f'\nMacro AUC: {np.mean(macro_means):.4f} ± {np.std(macro_means):.4f}')
print('\nNote: Fracture = NaN — no positive cases in the expert-labeled validation set.')

# Save summary
results_df.to_csv(CKPT_DIR / 'baseline_results.csv')
print(f'Saved to {CKPT_DIR / "baseline_results.csv"}')
